# Whom does a detail belong to? Representations of a small language model

The script runs from the project folder with the two data sets in `quellen_tab/` (Text Anonymization Benchmark, test split, Pilán et al. 2022) and `quellen_litbank/` (LitBank, Bamman et al. 2020). Model passes are cached in `outputs_notebook/`; with `RECOMPUTE = True` they are recomputed.

The model is Qwen2.5-1.5B. Unless stated otherwise, a span is represented by the mean of its token states at layer 16. Designs and comparison rules were fixed before each test.

## 0. Setup

In [1]:
import json, re, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, torch, spacy
from sklearn.cluster import DBSCAN
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from wordfreq import zipf_frequency
from transformers import AutoTokenizer, AutoModelForCausalLM

FOLDER = Path(".")
CACHE = FOLDER / "outputs_notebook"; CACHE.mkdir(exist_ok=True)
RECOMPUTE = False                      # True: recompute the model passes
MODEL = "Qwen/Qwen2.5-1.5B"
ATTENTION_LAYERS = [2, 8, 10, 14]      # fixed before the attention test
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 70)

In [2]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32, attn_implementation="eager").cuda().eval()
HOOKS = []                             # removed in Part 3, which needs no attention weights

def token_in_range(offsets, a, b):
    """Indices of the tokens that overlap the character range [a, b)."""
    return [i for i, (s, e) in enumerate(offsets) if e > s and s < b and e > a]

class Recorder:
    """Stores value vectors and attention weights of selected layers.

    contribution(target) returns, for every token j, the size of its direct contribution to position target,
    ||sum_h alpha_h[target, j] * W_O,h * v_h[j]|| (Kobayashi et al. 2020), summed over the layers.
    This is more precise than the attention weight alone, because it also accounts for what is passed on."""
    def __init__(self, model, layers):
        self.v, self.a, self.layers, self.model = {}, {}, layers, model
        for l in layers:
            at = model.model.layers[l].self_attn
            HOOKS.append(at.v_proj.register_forward_hook(lambda m, i, o, l=l: self.v.__setitem__(l, o.detach())))
            HOOKS.append(at.register_forward_hook(lambda m, i, o, l=l: self.a.__setitem__(l, o[1].detach())))
    def contribution(self, target):
        total = 0
        for l in self.layers:
            at = self.model.model.layers[l].self_attn
            alpha = self.a[l][0, :, target]; H, T = alpha.shape; hd = at.head_dim
            v = self.v[l][0].view(T, -1, hd); v = v.repeat_interleave(H // v.shape[1], dim=1)
            Wo = at.o_proj.weight.view(-1, H, hd)
            total = total + torch.einsum("thd,ehd->te", (alpha.T[:, :, None] * v).float(), Wo.float()).norm(dim=-1)
        return total.cpu().numpy()

recorder = Recorder(model, ATTENTION_LAYERS)

## 1. Does the attention from a person point to the details that identify them?

The 127 judgments of the TAB test split contain spans that annotators marked as direct identifiers, quasi-identifiers or harmless. A span counts as a quasi-identifier if most annotators marked it so. For every span before a mention of "applicant", I took the direct contribution of the span to the representation of "applicant" (attention weight times value vector, layers 2, 8, 10 and 14), as a share of the contribution of all earlier tokens. Judgments are long, so they are processed in pieces of 1,600 characters.

The measure is the AUC for quasi-identifier against harmless (0.5 is chance), overall and within the same kind of span (dates, places, organisations ...). Word frequency, surprisal, length and closeness to the next mention are the comparisons.

In [3]:
def spans_with_label(doc):
    votes, info = defaultdict(list), {}
    for annot in doc["annotations"].values():
        for m in annot["entity_mentions"]:
            k = (m["start_offset"], m["end_offset"]); votes[k].append(m["identifier_type"]); info[k] = (m["entity_type"], m["span_text"])
    rows = []
    for (a, b), v in votes.items():
        masked = sum(x in ("DIRECT", "QUASI") for x in v) / len(v) > 0.5
        label = "DIRECT" if v.count("DIRECT") > len(v) / 2 else ("QUASI" if masked else "NO_MASK")
        rows.append(dict(start=a, end=b, type=info[(a, b)][0], text=info[(a, b)][1], label=label, annotators=len(v)))
    return rows

def pieces(text, length=1600):
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + length)
        if end < len(text):
            nl = text.rfind("\n", start + length // 2, end); end = nl if nl > 0 else end
        out.append((start, end)); start = end
    return out

def zipf(s):
    w = re.findall(r"[A-Za-z]+", s)
    return float(np.mean([zipf_frequency(x, "en") for x in w])) if w else np.nan

@torch.no_grad()
def tab_pass(documents):
    rows = []
    for di, doc in enumerate(documents):
        text, spans = doc["text"], spans_with_label(doc)
        for c0, c1 in pieces(text):
            t = text[c0:c1]
            anchors = [m.start() for m in re.finditer(r"\bapplicant", t)]
            sp = [s for s in spans if s["start"] >= c0 and s["end"] <= c1
                  and any(a >= s["end"] - c0 for a in anchors) and not any(s["start"] - c0 <= a < s["end"] - c0 for a in anchors)]
            if not sp: continue
            enc = tok(t, return_offsets_mapping=True, return_tensors="pt"); off = enc.pop("offset_mapping")[0].tolist()
            ids = enc["input_ids"].cuda(); logp = model(input_ids=ids).logits[0].float().log_softmax(-1)
            content = [i for i, (s, e) in enumerate(off) if e > s]; sink = content[0]
            anchor_tok = sorted({token_in_range(off, a, a + 9)[0] for a in anchors})
            B = {ti: recorder.contribution(ti) for ti in anchor_tok}
            for s in sp:
                st = token_in_range(off, s["start"] - c0, s["end"] - c0)
                if not st: continue
                later = [ti for ti in anchor_tok if ti > max(st)]
                shares = [B[ti][st].sum() / B[ti][[j for j in content if j < ti and j != sink]].sum() / len(st) for ti in later]
                sur = [-float(logp[i - 1, ids[0, i]]) for i in st if i > 0]
                rows.append(dict(doc=doc["doc_id"], **s, attention=float(np.mean(shares)), word_frequency=zipf(s["text"]),
                                 surprisal=float(np.mean(sur)) if sur else np.nan,
                                 distance=min(later) - max(st), length=len(st)))
        if di % 25 == 0: print(f"{di} of {len(documents)} judgments", flush=True)
    return pd.DataFrame(rows)

GERMAN_COLUMNS = {"dok": "doc", "ende": "end", "typ": "type", "urteil": "label", "annotierende": "annotators", "aufmerksamkeit": "attention",
                  "wortfrequenz": "word_frequency", "ueberraschung": "surprisal", "abstand": "distance", "laenge": "length", "stueck": "piece"}
cache = CACHE / "tab_textstellen.csv"      # shared with the German notebook (German column names are translated on loading)
if RECOMPUTE or not cache.exists():
    documents = json.load(open(FOLDER / "quellen_tab/echr_test.json", encoding="utf-8"))
    tab = tab_pass(documents); tab.rename(columns={v: k for k, v in GERMAN_COLUMNS.items()}).to_csv(cache, index=False)
else:
    tab = pd.read_csv(cache).rename(columns=GERMAN_COLUMNS)
print(len(tab), "spans"); tab.label.value_counts()

7445 spans


label
QUASI      4273
NO_MASK    2864
DIRECT      308
Name: count, dtype: int64

In [4]:
MEASURES = {"attention": 1, "word_frequency": -1, "surprisal": 1, "distance": -1, "length": 1}   # sign: higher = more likely redacted
LABELS = {"attention": "Attention of the person (per token)", "word_frequency": "Word frequency (rare = redacted)",
          "surprisal": "Model surprisal", "distance": "Control: closeness to next mention", "length": "Control: length"}

def auc(d, measure):
    d = d.dropna(subset=[measure]); y = (d.label == "QUASI").astype(int)
    return roc_auc_score(y, MEASURES[measure] * d[measure]) if y.nunique() == 2 else np.nan

q = tab[tab.label.isin(["QUASI", "NO_MASK"])]
kinds = [a for a, g in q.groupby("type") if (g.label == "QUASI").sum() >= 20 and (g.label == "NO_MASK").sum() >= 20]
rows = []
for m in MEASURES:
    z = {"Measure": LABELS[m], "AUC overall": auc(q, m)}
    for a in kinds: z[f"AUC {a}"] = auc(q[q.type == a], m)
    z["AUC mean within kinds"] = np.nanmean([z[f"AUC {a}"] for a in kinds])
    rows.append(z)
result1 = pd.DataFrame(rows).set_index("Measure").round(2)
print("Count per kind:", {a: dict(q[q.type == a].label.value_counts()) for a in kinds})
result1

Count per kind: {'DATETIME': {'QUASI': np.int64(2082), 'NO_MASK': np.int64(178)}, 'DEM': {'NO_MASK': np.int64(441), 'QUASI': np.int64(232)}, 'LOC': {'QUASI': np.int64(291), 'NO_MASK': np.int64(178)}, 'MISC': {'NO_MASK': np.int64(311), 'QUASI': np.int64(214)}, 'ORG': {'NO_MASK': np.int64(1606), 'QUASI': np.int64(457)}, 'PERSON': {'QUASI': np.int64(692), 'NO_MASK': np.int64(90)}, 'QUANTITY': {'QUASI': np.int64(166), 'NO_MASK': np.int64(51)}}


,AUC overall,AUC DATETIME,AUC DEM,AUC LOC,AUC MISC,AUC ORG,AUC PERSON,AUC QUANTITY,AUC mean within kinds
Measure,,,,,,,,,
Attention of the person (per token),0.38,0.68,0.43,0.58,0.55,0.41,0.37,0.44,0.49
Word frequency (rare = redacted),0.56,0.68,0.54,0.63,0.50,0.81,0.60,0.64,0.63
Model surprisal,0.50,0.28,0.56,0.67,0.60,0.65,0.59,0.48,0.55
Control: closeness to next mention,0.54,0.64,0.42,0.48,0.53,0.56,0.56,0.45,0.52
Control: length,0.73,0.85,0.65,0.60,0.42,0.64,0.67,0.61,0.63


The attention of the person does not separate identifying from harmless details, overall or within a kind. Length alone separates them better. The first assumption did not hold.

## 2. Can the detail's own representation show whether it must be redacted?

Research on binding suggests that language models give an entity and its attributes a shared binding ID, a vector added to their representations (Feng & Steinhardt 2024). The information would then sit in the representations of the mentions themselves. I therefore trained a logistic regression on the representation of every annotated mention (first annotator; to be redacted = direct or quasi-identifier), cross-validated by judgment, so that each judgment is predicted by a model that has not seen it. This is the model that produces the boxes in the letters.

In [5]:
@torch.no_grad()
def tab_mentions(documents, with_representations):
    """All mentions of the first annotator, in text pieces with at least two mentions."""
    meta, reps = [], []
    for di, doc in enumerate(documents):
        ann = list(doc["annotations"].values())[0]["entity_mentions"]
        for ci, (c0, c1) in enumerate(pieces(doc["text"])):
            ms = sorted([m for m in ann if m["start_offset"] >= c0 and m["end_offset"] <= c1], key=lambda m: m["start_offset"])
            if len(ms) < 2: continue
            enc = tok(doc["text"][c0:c1], return_offsets_mapping=True, return_tensors="pt"); off = enc.pop("offset_mapping")[0].tolist()
            if with_representations:
                h = model(input_ids=enc["input_ids"].cuda(), output_hidden_states=True).hidden_states[16][0].float().cpu().numpy()
            for m in ms:
                st = token_in_range(off, m["start_offset"] - c0, m["end_offset"] - c0)
                if not st: continue
                if with_representations: reps.append(h[st].mean(0))
                meta.append(dict(doc=doc["doc_id"], entity=m["entity_id"], type=m["entity_type"], text=m["span_text"], ident=m["identifier_type"]))
        if with_representations and di % 25 == 0: print(f"{di} of {len(documents)} judgments", flush=True)
    return pd.DataFrame(meta), (np.stack(reps) if reps else None)

documents = json.load(open(FOLDER / "quellen_tab/echr_test.json", encoding="utf-8"))
cache_t = FOLDER / "outputs_pilot12/mention_reps_L16.npy"   # saved by the pilot; otherwise computed here
if RECOMPUTE or not cache_t.exists():
    cache_t = CACHE / "tab_mentions_L16.npy"
    if RECOMPUTE or not cache_t.exists(): TM, TR = tab_mentions(documents, True); np.save(cache_t, TR)
TM, _ = tab_mentions(documents, False); TR = np.load(cache_t).astype(np.float32); assert len(TM) == len(TR)
TM["redact"] = TM.ident.isin(["DIRECT", "QUASI"]).astype(int)
print(len(TM), "mentions,", TM.entity.nunique(), "entities;", TM.ident.value_counts().to_dict())

7364 mentions, 5549 entities; {'QUASI': 4790, 'NO_MASK': 2209, 'DIRECT': 365}


In [6]:
# out-of-fold predictions, five folds by judgment
TM["p"] = 0.0
for tr, te in GroupKFold(5).split(TR, TM.redact, TM.doc):
    inner = list(GroupKFold(4).split(TR[tr], TM.redact.values[tr], TM.doc.values[tr]))
    clf = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=[1e-4, 1e-3, 1e-2, 1e-1], cv=inner, scoring="roc_auc", max_iter=3000))
    TM.loc[TM.index[te], "p"] = clf.fit(TR[tr], TM.redact.values[tr]).predict_proba(TR[te])[:, 1]
print("AUC on judgments the model had not seen:", round(roc_auc_score(TM.redact, TM.p), 2))

AUC on judgments the model had not seen: 0.94


On judgments it had not seen, the model separates what must be redacted from what may stay with an AUC of 0.94.

## 3. Codes that link the mentions of one person

To test whether the representations show which mentions belong to the same person, I trained a linear projection to short binary codes (16 bits) on LitBank, 100 novel excerpts in which annotators marked which mentions refer to which entity. Mentions of the same entity are pulled below 2.5 bits of Hamming distance, mentions of different entities pushed above 5.5 bits. Within a text, mentions whose codes differ in at most one bit are grouped (DBSCAN). The groups are compared with the annotated entities by B³ F1 per text, against no grouping and grouping by identical wording, and separately for pronouns (PRON), descriptions (NOM) and names (PROP). The codes are evaluated in five folds by text.

In [7]:
for h in HOOKS: h.remove()
HOOKS.clear(); recorder.v.clear(); recorder.a.clear(); torch.cuda.empty_cache()
BITS = 16

def train_codes(groups, epochs, beta, W=None, seed=0):
    """Margin loss on a soft Hamming distance d = (16 - z_i.z_j) / 2 with z = tanh(beta * W x):
    same entity pulled below 2.5 bits, different entities pushed above 5.5 bits, plus 0.1 * mean((|z| - 1)^2)."""
    torch.manual_seed(seed)
    if W is None: W = torch.nn.Linear(model.config.hidden_size, BITS).cuda()
    opt = torch.optim.Adam(W.parameters(), lr=1e-3, weight_decay=1e-4)
    for ep in range(epochs):
        b = beta(ep)
        for Xg, eg in groups:
            if len(eg) < 2: continue
            same = eg[:, None] == eg[None, :]; off_diag = ~torch.eye(len(eg), dtype=bool, device="cuda")
            z = torch.tanh(b * W(Xg)); d = (BITS - z @ z.T) / 2
            pos = torch.relu(d - 2.5)[off_diag & same]; neg = torch.relu(5.5 - d)[off_diag & ~same]
            loss = ((pos ** 2).mean() if len(pos) else 0 * d.sum()) + ((neg ** 2).mean() if len(neg) else 0) + 0.1 * ((z.abs() - 1) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return W

def training_groups(M, R, stats, doc_col, docs):
    mu, sd = stats
    return [(torch.tensor((R[g.index.values] - mu) / sd).cuda(), torch.tensor(pd.factorize(pd.Series(g.entity.values))[0]).cuda())
            for _, g in M[M[doc_col].isin(docs)].groupby(doc_col)]

def binary_codes(W, R, stats):
    with torch.no_grad(): Z = torch.sign(W(torch.tensor((R - stats[0]) / stats[1]).cuda())).cpu().numpy()
    Z[Z == 0] = 1; return Z

def code_groups(Z, M, doc_col, docs):
    """DBSCAN on Hamming distance within each text: mentions whose codes differ in at most 1 bit are linked."""
    lab = pd.Series(index=M.index, dtype=object)
    for doc, g in M[M[doc_col].isin(docs)].groupby(doc_col):
        C = Z[g.index.values]; D = (C[:, None, :] != C[None, :, :]).sum(-1).astype(float)
        lab[g.index] = [f"{doc}:{x}" for x in DBSCAN(eps=1.5, min_samples=1, metric="precomputed").fit_predict(D)]
    return lab

def b3_per_mention(M, lab, doc_col):
    """B³ precision and recall of every mention against the reference entities (within its text)."""
    P, R = pd.Series(index=M.index, dtype=float), pd.Series(index=M.index, dtype=float)
    for _, g in M.assign(l=lab).groupby(doc_col):
        gold, pred = defaultdict(set), defaultdict(set)
        for i, e, l in zip(g.index, g.entity, g.l): gold[e].add(i); pred[l].add(i)
        for i, e, l in zip(g.index, g.entity, g.l):
            P[i] = len(gold[e] & pred[l]) / len(pred[l]); R[i] = len(gold[e] & pred[l]) / len(gold[e])
    return P, R

def b3(M, lab, doc_col, subset=None):
    """B³ F1, averaged per text as in the pilots; optionally only over a subset of mentions."""
    P, R = b3_per_mention(M, lab, doc_col)
    if subset is not None: P, R, M = P[subset], R[subset], M[subset]
    f = [2 * p * r / (p + r) for p, r in zip(P.groupby(M[doc_col]).mean(), R.groupby(M[doc_col]).mean())]
    return float(np.mean(f))

def reference_groups(M, doc_col, kind, wording=None):
    if kind == "none": return pd.Series([f"m{i}" for i in M.index], index=M.index)
    if kind == "same wording": return M[doc_col] + ":" + wording.str.strip().str.lower()
    return M[doc_col] + ":" + M.entity.astype(str)

In [8]:
LITBANK = FOLDER / "quellen_litbank/brat"
KIND = re.compile(r"^(PROP|NOM|PRON)_(PER|FAC|LOC|GPE|ORG|VEH)$")

def litbank_mentions():
    rows = []
    for ann in sorted(LITBANK.glob("*.ann")):
        doc = ann.stem; kinds_, clusters = {}, {}
        for line in ann.read_text(encoding="utf-8").splitlines():
            p = line.split("\t")
            if p[0].startswith("T"):
                lab, a, b = p[1].split(" ")[:3]
                (kinds_ if KIND.match(lab) else clusters)[(int(a), int(b))] = lab
        for (a, b), lab in kinds_.items():
            k, c = lab.split("_")
            rows.append(dict(doc=doc, start=a, end=b, kind=k, cls=c, entity=f"{doc}:{clusters.get((a, b), f'single_{a}_{b}')}"))
    return pd.DataFrame(rows).sort_values(["doc", "start", "end"]).reset_index(drop=True)

@torch.no_grad()
def litbank_representations(M):
    R = np.zeros((len(M), model.config.hidden_size), np.float32)
    for di, (doc, g) in enumerate(M.groupby("doc", sort=False)):
        text = (LITBANK / f"{doc}.txt").read_text(encoding="utf-8")
        enc = tok(text, return_offsets_mapping=True, return_tensors="pt"); off = enc.pop("offset_mapping")[0].tolist()
        h = model(input_ids=enc["input_ids"].cuda(), output_hidden_states=True).hidden_states[16][0]
        for i, a, b in zip(g.index, g.start, g.end): R[i] = h[token_in_range(off, a, b)].mean(0).float().cpu().numpy()
        del h; torch.cuda.empty_cache()
        if di % 20 == 0: print(f"{di} of 100 texts", flush=True)
    return R

LM = litbank_mentions()
LM["text"] = [(LITBANK / f"{d}.txt").read_text(encoding="utf-8")[a:b] for d, a, b in zip(LM.doc, LM.start, LM.end)]
cache_l = FOLDER / "outputs_pilot15/means_L16.npy"     # saved by the pilot (half-precision pass); otherwise computed here
if RECOMPUTE or not cache_l.exists():
    cache_l = CACHE / "litbank_L16.npy"
    if RECOMPUTE or not cache_l.exists(): np.save(cache_l, litbank_representations(LM).astype(np.float16))
LR = np.load(cache_l).astype(np.float32)
print(len(LM), "mentions in", LM.doc.nunique(), "texts;", LM.kind.value_counts().to_dict())

29103 mentions in 100 texts; {'PRON': 15816, 'NOM': 9737, 'PROP': 3550}


In [9]:
lit_docs = LM.doc.unique(); lit_folds = list(GroupKFold(5).split(lit_docs, groups=lit_docs))
lit_codes = pd.Series(index=LM.index, dtype=object)
for k, (tr, te) in enumerate(lit_folds):
    idx = LM.index[LM.doc.isin(lit_docs[tr])].values; stats = (LR[idx].mean(0), LR[idx].std(0) + 1e-6)
    W = train_codes(training_groups(LM, LR, stats, "doc", lit_docs[tr]), 300, lambda ep: 1 + 9 * ep / 299)
    lit_codes.update(code_groups(binary_codes(W, LR, stats), LM, "doc", lit_docs[te]))
    print("fold", k, "done", flush=True)

lit_groupings = {"no grouping": reference_groups(LM, "doc", "none"), "same wording": reference_groups(LM, "doc", "same wording", LM.text),
                 "codes (16 bits, DBSCAN 1 bit)": lit_codes}
pd.DataFrame({k: {"B³ all mentions": b3(LM, l, "doc"), **{f"B³ only {kind}": b3(LM, l, "doc", LM.kind == kind) for kind in ["PRON", "NOM", "PROP"]}}
              for k, l in lit_groupings.items()}).T.round(3)

fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done


,B³ all mentions,B³ only PRON,B³ only NOM,B³ only PROP
no grouping,0.426,0.171,0.721,0.420
same wording,0.567,0.425,0.752,0.593
"codes (16 bits, DBSCAN 1 bit)",0.685,0.652,0.721,0.666


For pronouns the codes group clearly better than identical wording or no grouping. For names they are somewhat better than identical wording, for descriptions not.

## 4. The same codes on court judgments

The codes trained on all LitBank texts are applied unchanged to the TAB judgments (standardised with the judgments' own statistics) and compared with TAB's entity annotation.

In [10]:
lit_stats, tab_stats = (LR.mean(0), LR.std(0) + 1e-6), (TR.mean(0), TR.std(0) + 1e-6)
W_lit = train_codes(training_groups(LM, LR, lit_stats, "doc", lit_docs), 300, lambda ep: 1 + 9 * ep / 299)
tab_docs = TM.doc.unique()
tab_groupings = {"no grouping": reference_groups(TM, "doc", "none"), "same wording": reference_groups(TM, "doc", "same wording", TM.text),
                 "codes from LitBank": code_groups(binary_codes(W_lit, TR, tab_stats), TM, "doc", tab_docs)}
pd.Series({k: b3(TM, l, "doc") for k, l in tab_groupings.items()}, name="B³ against TAB's groups").round(3)

no grouping           0.874
same wording          0.975
codes from LitBank    0.824
Name: B³ against TAB's groups, dtype: float64

In the judgments the codes are worse than identical wording, because names are repeated verbatim. A second difference is context: the judgments were processed in pieces of 1,600 characters, the novels in one pass, so the codes had to match mentions across pieces. The comparison therefore mixes the kind of text with missing context.

## 5. The two letters

Both letters are outside both training domains: first person and many pronouns. spaCy (`en_core_web_sm`) finds every noun chunk and every pronoun in the main text, between the salutation and the closing; nothing is chosen by hand. A chunk that starts with a possessive ("my internship") is split into the pronoun and the rest. The model reads the whole letter for context. Each span gets a box if the model from section 2, now trained on all judgments, gives it a probability of at least 0.5, and a group from the codes of section 4.

There is no annotated grouping for the letters, so the measures follow from the rule: how many groups my first-person pronouns fall into, whether spans after "my" join that group, and how many spans the redaction model would black out. The letter texts are not part of the repository; this cell reads them from `briefe/` if present and shows only counts.

In [11]:
nlp = spacy.load("en_core_web_sm")
FIRST, SECOND = {"i", "me", "my", "mine", "myself"}, {"you", "your", "yours", "yourself"}
LETTER_FILES = {"Saxion": FOLDER / "briefe/letter_saxion_v3_en.txt", "EECC": FOLDER / "briefe/letter_eecc_v10_en.txt"}

def letter_spans(text):
    b0 = text.index("\n", text.index("\nDear ") + 1) + 1; b1 = text.index("\nKind regards")
    doc, spans = nlp(text), []
    for ch in doc.noun_chunks:
        if ch.start_char < b0 or ch.end_char > b1: continue
        toks = list(ch)
        if len(toks) > 1 and toks[0].tag_ == "PRP$":
            spans.append((toks[0].idx, toks[0].idx + len(toks[0].text), "possessive"))
            rest = [t for t in toks[1:] if not t.is_punct]
            if rest: spans.append((rest[0].idx, rest[-1].idx + len(rest[-1].text), "owned_by_" + toks[0].lower_))
        else:
            spans.append((ch.start_char, ch.end_char, "pronoun" if len(toks) == 1 and toks[0].pos_ == "PRON" else "noun"))
    covered = np.zeros(len(text), bool)
    for a, b, _ in spans: covered[a:b] = True
    for t in doc:
        if t.pos_ == "PRON" and b0 <= t.idx < b1 and not covered[t.idx]: spans.append((t.idx, t.idx + len(t.text), "pronoun"))
    S = pd.DataFrame(spans, columns=["start", "end", "role"]).sort_values("start").reset_index(drop=True)
    S["text"] = [text[a:b] for a, b in zip(S.start, S.end)]
    S["person"] = ["first" if t.lower() in FIRST else "second" if t.lower() in SECOND else "" for t in S.text]
    return S

@torch.no_grad()
def span_representations(text, S):
    enc = tok(text, return_offsets_mapping=True, return_tensors="pt"); off = enc.pop("offset_mapping")[0].tolist()
    h = model(input_ids=enc["input_ids"].cuda(), output_hidden_states=True).hidden_states[16][0].float().cpu().numpy()
    return np.stack([h[token_in_range(off, a, b)].mean(0) for a, b in zip(S.start, S.end)])

probe_all = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=[1e-4, 1e-3, 1e-2, 1e-1], cv=list(GroupKFold(5).split(TR, TM.redact, TM.doc)),
                                                                 scoring="roc_auc", max_iter=3000)).fit(TR, TM.redact)
rows = {}
for name, path in LETTER_FILES.items():
    if not path.exists(): print(name, "letter not found, skipped"); continue
    text = re.sub("[¹²³⁴⁵⁶]", "", path.read_text(encoding="utf-8").split("\n---\n")[0])
    S = letter_spans(text); X = span_representations(text, S)
    Z = binary_codes(W_lit, X, lit_stats); D = (Z[:, None] != Z[None]).sum(-1).astype(float)
    S["group"] = DBSCAN(eps=1.5, min_samples=1, metric="precomputed").fit_predict(D)
    S["p"] = probe_all.predict_proba(X)[:, 1]
    first = S[S.person == "first"]; mine = first.group.value_counts().index[0]
    owned = S[S.role == "owned_by_my"]
    rows[name] = {"spans rated": len(S), "first-person pronouns": len(first), "groups they fall into": first.group.nunique(),
                  "share in the largest group": round((first.group == mine).mean(), 2),
                  "'my X' spans in that group": f"{(owned.group == mine).sum()} of {len(owned)}",
                  "boxed: first-person pronouns": round((first.p >= 0.5).mean(), 2),
                  "boxed: all other spans": round((S[S.person == ""].p >= 0.5).mean(), 2)}
pd.DataFrame(rows)

,Saxion,EECC
spans rated,192,171
first-person pronouns,23,28
groups they fall into,3,6
share in the largest group,0.91,0.82
'my X' spans in that group,0 of 8,1 of 9
boxed: first-person pronouns,0.57,0.61
boxed: all other spans,0.29,0.26


In both letters most of my first-person pronouns fall into one group, so the codes carry a usable signal for whom a pronoun refers to. None or almost none of the spans after "my" (internship, papers, skills, studies) joins that group: the codes link mentions of the same person, not the details that belong to the person. The redaction model, which works on judgments it has not seen, boxes more than half of my pronouns and also harmless words. A good result on court judgments says little about letters.

What remains open is a training target for ownership rather than identity, a step that finds mentions without a parser, a comparison with an existing coreference system, and linking across documents.

## References

Bamman, D., Lewke, O., & Mansoor, A. (2020). An annotated dataset of coreference in English literature. LREC 2020.
Feng, J., & Steinhardt, J. (2024). How do language models bind entities in context? ICLR 2024. https://arxiv.org/abs/2310.17191
Pilán, I., Lison, P., Øvrelid, L., Papadopoulou, A., Sánchez, D., & Batet, M. (2022). The Text Anonymization Benchmark (TAB). Computational Linguistics, 48(4). https://arxiv.org/abs/2202.00443